In [ ]:
%matplotlib inline

In [ ]:
import numpy as np
import cv2
import numpy as np
import glob
from PIL import Image, ImageDraw
# import some common detectron2 utilities
import detectron2
from detectron2.utils.logger import setup_logger
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog,DatasetCatalog
from detectron2.data import build_detection_train_loader, build_detection_test_loader
from detectron2.structures import BoxMode
from detectron2.data import detection_utils as utils
from detectron2.utils.visualizer import ColorMode
from detectron2.data import transforms as T
from detectron2.data import detection_utils as utils
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import torch
import copy
from detectron2.data import detection_utils as utils

In [ ]:
def get_object_detector(model_path, threshold):    
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("Misc/cascade_mask_rcnn_X_152_32x8d_FPN_IN5k_gn_dconv.yaml")) #Get the basic model configuration from the model zoo 
    cfg.DATASETS.TEST = ()
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 4 
    cfg.MODEL.WEIGHTS = model_path
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = threshold
    cfg.MODEL.MASK_ON = False

    model = build_model(cfg)
    model.eval()
    checkpointer = DetectionCheckpointer(model)
    checkpointer.load(cfg.MODEL.WEIGHTS)
    
    return model

def get_singlescale_detections(image, model):
    
    image = copy.deepcopy(image)
    h,w,_ = image.shape
    
    image = torch.as_tensor(image.transpose(2, 0, 1).astype("float32"))
    inputs = {"image": image, "height": h, "width": w}        
    with torch.no_grad():
        outputs = model([inputs])[0]
        final_boxes = outputs['instances'].pred_boxes.tensor.cpu().numpy().astype(int)
        final_scores = outputs['instances'].scores.cpu().numpy()
        final_classes = outputs['instances'].pred_classes.cpu().numpy()
    
    return final_boxes, final_classes, final_scores

def draw_buffer_frame_pred(bbox_pred, image):
    draw = ImageDraw.Draw(image)
    draw.rectangle([(bbox_pred[0],bbox_pred[1]),(bbox_pred[2], bbox_pred[3])], outline='blue', width=3)   
        
    return image

def preannotate_image(im,model_od):        
    im_rgba = Image.fromarray(im).convert('RGBA')

    with torch.no_grad():  
        pred_od, classes_od, scores_od = get_singlescale_detections(im, model_od)

    for i in range(len(pred_od)):
        im_rgba = draw_buffer_frame_pred(pred_od[i], im_rgba)

    #im_rgba.save(save_dir + '/' + str(counter).zfill(4) + '.png')

    return im_rgba

In [ ]:
threshold = 0.5
od_path = '/root/output/cascadeX52_training0/model_0019999.pth'
model_od =  get_object_detector(od_path, threshold)

In [ ]:
d = '/root/data/images/'
imgs = sorted(glob.glob(d+'*.jpeg'))

In [ ]:
image = utils.read_image(imgs[2], format="BGR")
im_rgba = preannotate_image(image, model_od)

In [ ]:
im_rgba